In [0]:
bronze_df_users = spark.read.table("adbanalitycs.bronze.users")
print(f"Total registros: {df_users.count()}")
#df = spark.read.table("bronze.users")
display(bronze_df_users.select("value"))

In [0]:
from pyspark.sql.functions import (split, col, size, lit, trim, lower, upper, when, regexp_replace, current_timestamp, expr)

df = bronze_df_users.withColumn("parts", split(col("value"), r"\|"))
valido_df_bronze_users = df.filter(size(col("parts")) == 8)
invalido_df_bronze_users = df.filter(size(col("parts")) !=8)

silver_df = valido_df_bronze_users.select(
    col("parts")[0].alias("user_id"),
    col("parts")[1].alias("full_name"),
    col("parts")[2].alias("document_id"),
    col("parts")[3].alias("email"),
    col("parts")[4].alias("phone"),
    col("parts")[5].alias("country"),
    col("parts")[6].alias("segment"),
    col("parts")[7].alias("registration_date")
)
print(f"Total registros validos de la tabla users: {silver_df.count()}")
#display(silver_df.limit(20))

quarantine_df = invalido_df_bronze_users \
    .withColumn("error_type",lit("STRUCTURE_MISMATCH")) \
    .withColumn("error_reason",lit("Invalid number of columns after split")) \
    .withColumn("_quarantine_timestamp",current_timestamp())

silver_df.write.mode("overwrite").saveAsTable("adbanalitycs.silver.users")

quarantine_df.withColumn("error_type", lit("DATA_QUALITY_ERROR")).write.mode("overwrite").saveAsTable("adbanalitycs.silver.users_quarantine")

#display(quarantine_df.limit(20))
#quarantine_df.write.mode("overwrite").saveAsTable("adbanalitycs.silver.users_quarantine")
#silver_df.write.mode("overwrite").saveAsTable("adbanalitycs.silver.users")




In [0]:
from pyspark.sql import functions as F


df_users_silver_validado = (

    silver_df

    # ========================================
    # LIMPIEZA BASE
    # ========================================

    .withColumn("user_id", trim(col("user_id")))
    .withColumn("full_name", trim(col("full_name")))
    .withColumn("document_id", trim(col("document_id")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("phone", trim(col("phone")))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("segment", lower(trim(col("segment"))))
    .withColumn("registration_date", trim(col("registration_date")))

    # ========================================
    # NORMALIZACIÓN COUNTRY
    # ========================================

    .withColumn(
        "country",
        when(col("country").isin("PERU", "PER"), "PE")
        .when(col("country").isin("CHILE", "CHI"), "CL")
        .when(col("country").isin("COLOMBIA", "COL"), "CO")
        .when(col("country").isin("ARGENTINA", "ARG"), "AR")
        .when(col("country").isin("MEXICO", "MEX"), "MX")
        .otherwise(col("country"))
    )

    # ========================================
    # NORMALIZACIÓN SEGMENT
    # ========================================

    .withColumn(
        "segment",
        when(col("segment").isin("premium"), "premium")
        .when(col("segment").isin("estandar"), "estandar")
        .when(col("segment").isin("nuevo"), "nuevo")
        .otherwise(col("segment"))
    )

    # ========================================
    # PARSE FECHA
    # ========================================

    .withColumn(
        "registration_date_parsed",
        when(
            col("registration_date").rlike(r"^[0-9]{2}/[0-9]{2}/[0-9]{4}$"),
            expr("try_to_timestamp(registration_date, 'dd/MM/yyyy')").cast("date")
        )
        .when(
            col("registration_date").rlike(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$"),
            expr("try_to_timestamp(registration_date, 'yyyy-MM-dd')").cast("date")
        )
        .otherwise(lit(None))
    )

    # ========================================
    # VALIDACIONES
    # ========================================

    .withColumn("is_valid_user_id", col("user_id").rlike("^USR-[0-9]{6}$"))
    .withColumn("is_valid_document_id", col("document_id").rlike("^[0-9]{7,9}$"))
    .withColumn("is_valid_email", col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))
    .withColumn("phone_has_letters", col("phone").rlike(r"[A-Za-z]"))
    .withColumn("phone_clean",when(~col("phone_has_letters"),regexp_replace(col("phone"),r"[^0-9]","")))
    .withColumn("is_valid_phone",col("phone_clean").rlike(r"^[0-9]{10,15}$"))
    .withColumn(
        "phone_country_code",
        when(col("phone_clean").startswith("51"),"PE")
        .when(col("phone_clean").startswith("56"),"CL")
        .when(col("phone_clean").startswith("57"),"CO")
        .when(col("phone_clean").startswith("54"),"AR")
        .when(col("phone_clean").startswith("52"),"MX")
        .otherwise("UNKNOWN")    
    )

    .withColumn(
        "phone_country_match",
        col("phone_country_code") == col("country")
    )

    .withColumn(
        "is_valid_phone_validation",
        (~col("phone_has_letters")) &
        (col("is_valid_phone")) &
        (col("phone_country_code") != "UNKNOWN") &
        (col("phone_country_match"))
    )

    .withColumn(
        "is_valid_country",
        col("country").isin(
            "PE",
            "CL",
            "CO",
            "AR",
            "MX"
        )
    )

    .withColumn(
        "is_valid_segment",
        (
            col("segment").isNull()
        ) |
        (
            col("segment").isin(
                "premium",
                "estandar",
                "nuevo"
            )
        )
    )

    .withColumn(
        "is_valid_registration_date",
        col("registration_date_parsed").isNotNull()
    )

)

display(df_users_silver_validado)

In [0]:
df_users_silver = (

    df_users

    # ========================================
    # LIMPIEZA BASE
    # ========================================

    .withColumn("user_id", trim(col("user_id")))
    .withColumn("full_name", trim(col("full_name")))
    .withColumn("document_id", trim(col("document_id")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("phone", trim(col("phone")))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("segment", lower(trim(col("segment"))))
    .withColumn("registration_date", trim(col("registration_date")))

    # ========================================
    # NORMALIZACIÓN COUNTRY
    # ========================================

    .withColumn(
        "country",
        when(col("country").isin("PERU", "PER"), "PE")
        .when(col("country").isin("CHILE", "CHI"), "CL")
        .when(col("country").isin("COLOMBIA", "COL"), "CO")
        .when(col("country").isin("ARGENTINA", "ARG"), "AR")
        .when(col("country").isin("MEXICO", "MEX"), "MX")
        .otherwise(col("country"))
    )

    # ========================================
    # NORMALIZACIÓN SEGMENT
    # ========================================

    .withColumn(
        "segment",
        when(col("segment").isin("premium"), "premium")
        .when(col("segment").isin("estandar"), "estandar")
        .when(col("segment").isin("nuevo"), "nuevo")
        .otherwise(col("segment"))
    )

    # ========================================
    # PARSE FECHA
    # ========================================

    .withColumn(
        "registration_date_parsed",
        when(
            col("registration_date").rlike(
                r"^[0-9]{2}/[0-9]{2}/[0-9]{4}$"
            ),
            expr(
                "try_to_timestamp(registration_date, 'dd/MM/yyyy')"
        ).cast("date")
        )
        .when(
            col("registration_date").rlike(
                r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$"
            ),
            expr(
                "try_to_timestamp(registration_date, 'yyyy-MM-dd')"
        ).cast("date")
        )
        .otherwise(lit(None))
    )

    # ========================================
    # VALIDACIONES
    # ========================================

    .withColumn(
        "is_valid_user_id",
        col("user_id").rlike("^USR-[0-9]{6}$")
    )

    .withColumn(
        "is_valid_document_id",
        col("document_id").rlike("^[0-9]{7,9}$")
    )

    .withColumn(
        "is_valid_email",
        col("email").rlike(
            #"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )
    )

    .withColumn(
        "phone_has_letters",
        col("phone").rlike(r"[A-Za-z]")
    )

    .withColumn(
        "phone_clean",
        when(
            ~col("phone_has_letters"),
            regexp_replace(col("phone"),r"[^0-9]",""
            )
        )
    )

    .withColumn(
        "is_valid_phone",
        col("phone_clean").rlike(
            r"^[0-9]{10,15}$"
        )
    )

    .withColumn(
        "phone_country_code",
        when(col("phone_clean").startswith("51"),"PE")
        .when(col("phone_clean").startswith("56"),"CL")
        .when(col("phone_clean").startswith("57"),"CO")
        .when(col("phone_clean").startswith("54"),"AR")
        .when(col("phone_clean").startswith("52"),"MX")
        .otherwise("UNKNOWN")    
    )

    .withColumn(
        "phone_country_match",
        col("phone_country_code") == col("country")
    )

    .withColumn(
        "is_valid_phone",
        (~col("phone_has_letters")) &
        (col("is_valid_phone")) &
        (col("phone_country_code") != "UNKNOWN") &
        (col("phone_country_match"))
    )

    .withColumn(
        "is_valid_country",
        col("country").isin(
            "PE",
            "CL",
            "CO",
            "AR",
            "MX"
        )
    )

    .withColumn(
        "is_valid_segment",
        (
            col("segment").isNull()
        ) |
        (
            col("segment").isin(
                "premium",
                "estandar",
                "nuevo"
            )
        )
    )

    .withColumn(
        "is_valid_registration_date",
        col("registration_date_parsed").isNotNull()
    )

)

display(df_users_silver)

In [0]:
display(
    df_users_silver_validado.select(
        count(
            when(~col("is_valid_user_id"), 1)
        ).alias("invalid_user_id"),

        count(
            when(~col("is_valid_document_id"), 1)
        ).alias("invalid_document_id"),

        count(
            when(~col("is_valid_email"), 1)
        ).alias("invalid_email"),

        count(
            when(~col("is_valid_phone"), 1)
        ).alias("invalid_phone"),

        count(
            when(~col("is_valid_country"), 1)
        ).alias("invalid_country"),

        count(
            when(~col("is_valid_segment"), 1)
        ).alias("invalid_segment"),

        count(
            when(~col("is_valid_registration_date"), 1)
        ).alias("invalid_registration_date")
    )
)




In [0]:

display(
    df_users_silver_validado.filter(
        col("is_valid_email") == False
    ).select("email") 
)


"""
display(

    df_users_silver
    .filter(
        col("is_valid_phone") == False
    )

    .withColumn(
        "phone_length",
        length(col("phone_clean"))
    )

    .select(
        "phone",
        "phone_clean",
        "country",
        "phone_country_code",
        "phone_length"
    )
    .orderBy(
        col("country"),
        col("phone_length")
    )
)
"""


In [0]:
# ============================================
# CLEAN RECORDS
# ============================================

df_users_clean = (
    df_users_silver.filter(

        col("is_valid_user_id") &
        col("is_valid_document_id") &
        col("is_valid_email") &
        col("is_valid_phone") &
        col("is_valid_country") &
        col("is_valid_segment") &
        col("is_valid_registration_date")
    )
)

# ============================================
# QUARANTINE RECORDS
# ============================================

df_users_quarantine = (
    df_users_silver.filter(

        ~(
            col("is_valid_user_id") &
            col("is_valid_document_id") &
            col("is_valid_email") &
            col("is_valid_phone") &
            col("is_valid_country") &
            col("is_valid_segment") &
            col("is_valid_registration_date")
        )
    )
)

print("Clean records:", df_users_clean.count())
print("Quarantine records:", df_users_quarantine.count())